# 006_postprocess_within_temporal_condition_specific.ipynb

Within-condition temporal postprocessing for the condition-specific ranking branch.


In [ ]:
from pathlib import Path

%matplotlib inline
import os, warnings
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.io import loadmat
from matplotlib import cm, colors as mcolors
try:
    import nibabel as nib
    from nilearn.input_data import NiftiMasker
    from nilearn import plotting as niplot
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False
print("Ready.")

In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "006_postprocess_within_temporal_refined"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None

    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None

    _fig_counter += 1

    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)

    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

def show_save_close(name=None):
    save_current_fig(name)
    plt.show()
    plt.close()

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
# ============================================================
# NOTEBOOK IDENTITY / SAVED DECODING SETTINGS
# ============================================================

ANALYSIS_TYPE = "temporal"
FIT_SCOPE = "within"

USE_SAVED_DECODING = True
DECODING_OUTPUT_DIR = "msaa_condrank_decoding_outputs_temporal_within"
PER_ARCH_DECODING_CSV = os.path.join(DECODING_OUTPUT_DIR, "per_archetype_mean_accuracy.csv")

USE_CACHE = True
OVERWRITE_CACHE = False

In [ ]:

FIT_LOAD_DIR = "msaa_flexible_outputs_npz"
DECODE_LOAD_DIR = "msaa_condrank_decoding_outputs_temporal_within"
CONDITIONS = ["intact", "word", "rest"]
K_VALUES = [10]
TOP_N_REPORT = 5

SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"
COND_COLORS = {"intact":"purple","word":"green","rest":"black"}
OUTPUT_DIR = "006_postprocess_within_temporal_condition_specific_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# LOAD SAVED PER-ARCHETYPE DECODING
# ============================================================

def load_per_archetype_decoding(path=PER_ARCH_DECODING_CSV):
    if not os.path.exists(path):
        print("Saved per-archetype decoding CSV not found:", path)
        return pd.DataFrame()

    df = pd.read_csv(path)
    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "sem" not in df.columns:
        rename["sem_accuracy"] = "sem"
    if "err" in df.columns and "sem" not in df.columns:
        rename["err"] = "sem"
    df = df.rename(columns=rename)

    if "analysis_type" in df.columns:
        df = df[df["analysis_type"].astype(str) == ANALYSIS_TYPE].copy()
    if "fit_scope" in df.columns:
        df = df[df["fit_scope"].astype(str) == FIT_SCOPE].copy()

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "archetype" in df.columns:
        df["archetype"] = df["archetype"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    print("Loaded saved per-archetype decoding:", path)
    print("Rows:", len(df))
    if len(df):
        display(df.head())
    return df

per_arch_decoding_df = load_per_archetype_decoding()

In [ ]:

def to_float_array(x): return np.array(x, dtype=float)
def load_msaa_npz(path):
    data=np.load(path, allow_pickle=True)
    results_subj=data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray): results_subj=results_subj.tolist()
    return {"K":int(data["K"]), "results_subj":results_subj}
fits={cond:{K:load_msaa_npz(os.path.join(FIT_LOAD_DIR, f"temporalAA_within_{cond}_K{K}.npz")) for K in K_VALUES} for cond in CONDITIONS}
rankings=np.load(os.path.join(DECODE_LOAD_DIR, "rankings.npy"), allow_pickle=True).item()
per_arch_df=pd.read_csv(os.path.join(DECODE_LOAD_DIR, "per_archetype_mean_accuracy.csv"))
posterior=loadmat(POSTERIOR_MAT)
centers=to_float_array(posterior['posterior']['centers'][0][0][0][0][0]); widths=to_float_array(list(posterior['posterior']['widths'][0][0][0][0][0][:,0].T)).ravel()
lookup_table={'Vis':'Visual','SomMot':'Somatomotor','DorsAttn':'Dorsal attention','SalVentAttn':'Ventral attention','Limbic':'Limbic','Cont':'Frontoparietal','Default':'Default mode'}
network_colors={'Visual':'#D7DF23','Somatomotor':'#39B54A','Dorsal attention':'#00A79D','Ventral attention':'#27AAE1','Limbic':'#1C75BC','Frontoparietal':'#92278F','Default mode':'#EE2A7B'}
network_codes={k:i+1 for i,k in enumerate(lookup_table.values())}
def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals=np.asmatrix(range(1,dims[0]+1)).T
        if len(dims)==1:return vals
        aftervals=np.asmatrix(fullfact(dims[1:]))
        inds=np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row=0
        for i in range(aftervals.shape[0]):
            inds[row:(row+len(vals)),0]=vals
            inds[row:(row+len(vals)),1:]=np.tile(aftervals[i,:], (len(vals),1))
            row+=len(vals)
        return inds
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img=nib.load(nifti_file) if type(nifti_file)==str else nifti_file
        mask=NiftiMasker(mask_strategy='background'); mask.fit(nifti_file if mask_file is None else mask_file)
    Saff=img.get_sform(); Y=np.float32(mask.transform(nifti_file)).copy()
    vmask=np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj,(1,np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords=fullfact(img.shape[0:3])[vmask, ::-1]-1
    R=np.array(np.dot(vox_coords,Saff[0:3,0:3]))+Saff[:3,3]
    return {'Y':Y,'R':R}
def rbf(R, center, width): return np.exp(-np.sum((R-center)**2, axis=1)/width)
def node_labels(centers, widths, networks_cmu):
    labels=[]
    for c,w in zip(centers,widths):
        r=rbf(networks_cmu['R'],c,w)
        label_weights=[sum(r[networks_cmu['Y'].ravel()==i]) for i in range(1,len(network_codes)+1)]
        labels.append(np.argmax(label_weights)+1)
    return pd.DataFrame({'code':labels,'Network':[list(lookup_table.values())[i-1] for i in labels]})
if NILEARN_AVAILABLE:
    key=pd.read_csv(SCHAEFER_TXT, sep='\t', header=None, names=['id','name','x','y','z','t']).drop('t', axis=1)
    key['network']=key['name'].apply(lambda x: lookup_table[x.split('_')[2]])
    key['code']=key['network'].apply(lambda x: network_codes[x]); key.set_index('id', inplace=True); key.loc[0,'code']=0
    networks_cmu=nii2cmu(SCHAEFER_NII); networks_cmu['Y']=np.atleast_2d(np.array([key.loc[i,'code'] for i in networks_cmu['Y']]).astype(float))
    node_code_df=node_labels(centers,widths,networks_cmu)
else:
    node_code_df=None
print("Loaded.")

In [ ]:

def get_decoding_value(cond_name, K, k):
    sub=per_arch_df[(per_arch_df["condition"]==cond_name) & (per_arch_df["K"]==K) & (per_arch_df["archetype"]==k)]
    return float(sub.iloc[0]["mean_accuracy"]) if len(sub) else None
def plot_decoding_bar(cond_name, K, k):
    val=get_decoding_value(cond_name, K, k)
    plt.figure(figsize=(4,4)); plt.bar([f"{cond_name}\nK={K}\na={k}"], [val], color=[COND_COLORS[cond_name]]); plt.ylabel("Mean decoding accuracy"); plt.title("Per-archetype decoding"); plt.tight_layout(); plt.show()
def plot_coeff_timecourse(results_subj, cond_name, K, k):
    Xk=np.stack([to_float_array(sub["S"])[k,:] for sub in results_subj], axis=0); mean=Xk.mean(axis=0); sem=Xk.std(axis=0)/np.sqrt(max(Xk.shape[0],1))
    x=np.arange(len(mean)); plt.figure(figsize=(9,4)); plt.plot(x, mean, color=COND_COLORS[cond_name]); plt.fill_between(x, mean-sem, mean+sem, color=COND_COLORS[cond_name], alpha=0.2); plt.title(f"{cond_name} | K={K} | archetype {k} | coefficient timecourse"); plt.tight_layout(); plt.show()
def plot_subject_heatmap(results_subj, cond_name, K, k):
    Xk=np.stack([to_float_array(sub["S"])[k,:] for sub in results_subj], axis=0)
    plt.figure(figsize=(10,5)); ax=sns.heatmap(Xk, cmap="coolwarm", center=0)
    for tick_label in ax.get_yticklabels(): tick_label.set_color(COND_COLORS[cond_name]); tick_label.set_fontsize(8)
    plt.title(f"{cond_name} | K={K} | archetype {k} | subject heatmap"); plt.tight_layout(); plt.show()
def plot_spatial_map(results_subj, cond_name, K, k):
    if not NILEARN_AVAILABLE: return
    vals=np.stack([to_float_array(sub["sXC"])[:,k] for sub in results_subj], axis=0).mean(axis=0); vals=np.nan_to_num(vals); vmax=np.max(np.abs(vals))
    norm=mcolors.TwoSlopeNorm(vmin=-vmax,vcenter=0.0,vmax=vmax); cmap=cm.get_cmap("coolwarm")
    node_colors=[cmap(norm(v)) for v in vals]; node_sizes=8+18*(np.abs(vals)/(vmax+1e-8))
    disp=niplot.plot_connectome(np.eye(centers.shape[0]), centers, node_color=node_colors, node_size=node_sizes, display_mode="lyrz", title=f"{cond_name} | K={K} | archetype {k} | signed spatial map")
    save_current_fig()
    plt.show()
    plt.close()
    try: disp.close()
    except Exception: pass
def plot_network_bar(results_subj, cond_name, K, k):
    if node_code_df is None: return
    vals=np.stack([to_float_array(sub["sXC"])[:,k] for sub in results_subj], axis=0).mean(axis=0)
    network_df=node_code_df.copy(); network_df["value"]=np.nan_to_num(vals)
    means=network_df.groupby("Network")["value"].mean().reindex(list(network_colors.keys()))
    plt.figure(figsize=(7.5,4)); plt.bar(means.index, means.values, color=[network_colors[n] for n in means.index]); plt.axhline(0, color="black"); plt.xticks(rotation=35, ha="right")
    plt.title(f"{cond_name} | K={K} | archetype {k} | network mean values"); plt.tight_layout(); plt.show()

for cond_name in CONDITIONS:
    for K in K_VALUES:
        results_subj=fits[cond_name][K]["results_subj"]; top_arches=list(rankings[(cond_name, K)][:TOP_N_REPORT])
        print(f"\nCondition={cond_name} | K={K} | top archetypes:", top_arches)
        for k in top_arches:
            plot_decoding_bar(cond_name, K, k); plot_coeff_timecourse(results_subj, cond_name, K, k); plot_subject_heatmap(results_subj, cond_name, K, k); plot_spatial_map(results_subj, cond_name, K, k); plot_network_bar(results_subj, cond_name, K, k)